# logit-thermo A2-8：ファントホッフの ΔH と熱量計の ΔH（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 30〜50 分。終わったら自動で切れます。
- ゲート：`docs/A2_8_GATE.md`。新しく生成はしません。A2-2 と A2-7b（なし・なし（別の種））で保存した生成のトークンを、
  Pythia-1B・OLMo-2-1B・Qwen2.5-1.5B に読み直させて、1 トークンごとのエネルギー（温度 1 の −log p）を出し、
  凍る速さ・溶ける速さと合わせて判定して `A2_8/a2_8_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2 のデータ集めのコードと結晶の始まりの探索のコードは Drive の前のノートブックから取り出してハッシュで確かめ、
  `scripts/a2_8_vh.py`（`06a93c4eec9f3e77…`）は埋め込み（分析の部分はデータを開く前に固定）。
- 保存先 `A2_8/`。途中で切れても、もう一度「すべて実行」すれば読み直しの終わったモデルは飛ばします。
- **通知**：🚀・材料ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-8 ファントホッフと熱量計（3 モデルの読み直し → 分析、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-8 ファントホッフと熱量計（3 モデルの読み直し → 分析、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-8 van t Hoff vs calorimetric', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import gc, math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9'})
SRC22, SRCON = _s['SRC22'], _s['SRCON']
SRC8 = '"""A2-8（ゲート docs/A2_8_GATE.md）：ファントホッフの ΔH と熱量計の ΔH。GPU（L4）で読み直し、そのあと同じノートブックで分析（CPU の計算）。\nscripts/a2_2_collect.py（load_model・valid_mask・step_stats・h2_of・CONT）と scripts/a2_explore_onset.py（copy_marks）を\n実行したのと同じ名前空間で実行する。分析の部分はデータを開く前に固定する。\n\n材料（新しく生成はしない）：\n- A2-2 の生成（温度 15 × サンプル 2 × 記事 40 × 512 歩）：`A2_2/{name}.npz` の toks。書き出しは A2-7b の npz の prompts を使う\n  （A2-7b は A2-2 と同じ記事・同じ切り方。use が同じことを確かめる）。\n- A2-7b の「なし」と「なし（別の種）」（温度 8 × サンプル 2 × 記事 40 × 512 歩）：`A2_7B/{name}@none.npz`・`@none_b.npz`。\n読み直し（teacher forcing）：書き出し ＋ 生成したトークンをモデルに通し、各歩で\n- U ＝ −log p₁(選んだトークン)（温度 1、使わない番号と EOS を外す。生成のときと同じ valid）\n- H1 ＝ 温度 1 の分布のエントロピー、HT ＝ 生成した温度の分布のエントロピー\n- ELL ＝ 生成した温度の ℓ（順位 H 以降）。保存してある ℓ と比べて、トークンと位置がずれていないことを確かめる。\n"""\nimport gc, math, os, time\nimport numpy as np\nimport torch\n\nA8_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')]\nA8_T22 = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nA8_T7B = [0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 1.0]\nA8_TU = sorted(set(A8_T22) | set(A8_T7B))                     # 17 の温度\nA8_SOURCES = [(\'a22\', \'A2_2/{name}.npz\', A8_T22), (\'b7n\', \'A2_7B/{name}@none.npz\', A8_T7B), (\'b7b\', \'A2_7B/{name}@none_b.npz\', A8_T7B)]\nA8_BS = 4\n# T_0 のものさし：A2-2 の結果（fb_H の T0 と 95% 区間）。A2-2 の生成とは同じデータから出ているが、決め方（ℓ の交点）は別。\nA8_T0 = {\'pythia-1b\': (0.852045, 0.810311, 0.869373), \'olmo2-1b\': (0.800822, 0.775764, 0.856064),\n         \'qwen2.5-1.5b\': (0.910086, 0.861524, 0.956799)}\n# 相の決め方\nA8_W = 32                 # 窓（中心をそろえる：歩 t の相は t−16〜t+15 の写しの印の平均で決める）\nA8_ENTER, A8_EXIT = 0.5, 0.25\nA8_SENS_EXIT = [0.5, 0.1]  # 記録用の別の出口\nA8_MIN_EV = 10            # 使う温度：凍る回と溶ける回が、それぞれこれ以上\nA8_MIN_TEMPS = 3\nA8_BOOT, A8_BOOT_SEED = 1000, 20261112\nA8_Q1_BAND = 1.10\nA8_Q2_BINS = [(0.5, \'1 より小さい（途中の段がある）\'), (2.0, \'1 くらい（ばらばらに溶ける）\'), (10.0, \'数トークン（2〜10）\'),\n              (100.0, \'数十トークン（10〜100、結晶核くらい）\'), (float(\'inf\'), \'100 トークン以上\')]\n\n\n# ---------- 読み直し（GPU） ----------\n\n@torch.no_grad()\ndef score_seqs(model, prompts, toks, temps, valid, H2, device):\n    """prompts：(N, P) の list、toks：(N, L)、temps：(N,)。→ U, H1, HT, ELL：(N, L) の float32。"""\n    N, L = toks.shape\n    out = {k: np.zeros((N, L), np.float32) for k in (\'U\', \'H1\', \'HT\', \'ELL\')}\n    for i0 in range(0, N, A8_BS):\n        ii = list(range(i0, min(N, i0 + A8_BS)))\n        P = len(prompts[ii[0]])\n        x = torch.tensor([list(prompts[i]) + list(map(int, toks[i])) for i in ii], device=device)\n        lg = model(x).logits[:, P - 1: P - 1 + L].float()                  # 歩 t の logit は toks[t] を選ぶ前のもの\n        for b, i in enumerate(ii):\n            z = lg[b]\n            y = torch.tensor(toks[i], device=device, dtype=torch.long)\n            z1 = z.double().masked_fill(~valid, float(\'-inf\'))\n            lp1 = torch.log_softmax(z1, dim=-1)\n            p1 = torch.exp(lp1)\n            out[\'U\'][i] = (-lp1.gather(-1, y[:, None])[:, 0]).float().cpu().numpy()\n            out[\'H1\'][i] = (-torch.where(p1 > 0, p1 * lp1, torch.zeros_like(p1)).sum(-1)).float().cpu().numpy()\n            ell, _, ent, _ = step_stats(z, float(temps[i]), valid, H2)\n            out[\'ELL\'][i] = ell.float().cpu().numpy(); out[\'HT\'][i] = ent.float().cpu().numpy()\n    return out\n\n\ndef load_sources(base, name):\n    """→ {source: dict(toks=(nT, S, nD, L), temps, stored_ell=(nT, S, nD, L))}, prompts（A2-7b の npz から）, use。"""\n    Zb = np.load(f\'{base}/A2_7B/{name}@none.npz\')\n    prompts = [list(map(int, p)) for p in Zb[\'prompts\']]\n    use = [int(u) for u in Zb[\'use\']]\n    src = {}\n    for key, pat, temps in A8_SOURCES:\n        Z = np.load(f\'{base}/\' + pat.format(name=name))\n        assert [float(t) for t in Z[\'temps\']] == temps, (name, key)\n        assert [int(u) for u in Z[\'use\']] == use, (\'記事の並びが違います\', name, key)\n        toks = Z[\'toks\']\n        ell = Z[\'gen\'][0] if key == \'a22\' else Z[\'g_ell\']\n        if key == \'a22\':\n            assert int(Z[\'prompt_len\']) == len(prompts[0]), (\'書き出しの長さが A2-2 と違います\', name, int(Z[\'prompt_len\']), len(prompts[0]))\n        assert toks.shape == ell.shape and toks.shape[2] == len(use), (toks.shape, ell.shape)\n        src[key] = dict(toks=toks, temps=temps, stored_ell=ell)\n    return src, prompts, use\n\n\ndef run_score(name, hf_id, base, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう読み直しが終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    src, prompts, use = load_sources(base, name)\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    save, chk = {}, {}\n    for key, d in src.items():\n        nT, S, nD, L = d[\'toks\'].shape\n        assert L == CONT\n        flat = d[\'toks\'].reshape(-1, L)\n        temps = np.repeat(np.array(d[\'temps\']), S * nD)\n        prm = [prompts[j] for _ in range(nT) for _ in range(S) for j in range(nD)]\n        r = score_seqs(model, prm, flat, temps, valid, H2, device)\n        diff = np.abs(r[\'ELL\'] - d[\'stored_ell\'].reshape(-1, L))\n        diff = diff[np.isfinite(diff)]\n        chk[key] = dict(med=float(np.median(diff)), q99=float(np.quantile(diff, 0.99)), mx=float(diff.max()))\n        # トークンと位置がずれていたら、ℓ は何単位も違う（数値のずれは 1/1000 くらい）\n        assert chk[key][\'med\'] < 0.05, (\'読み直しの ℓ が保存した ℓ と合いません（トークンか位置のずれ）\', name, key, chk[key])\n        for k, v in r.items():\n            if k != \'ELL\':\n                save[f\'{key}_{k}\'] = v.reshape(nT, S, nD, L)\n        save[f\'{key}_toks\'] = d[\'toks\']\n        notify(f\'{name}・{key} の読み直しおわり（{time.time() - t0:.0f} 秒、ℓ のずれの中央値 {chk[key]["med"]:.1e}）\')\n    np.savez_compressed(path + \'.part.npz\', use=np.array(use), check=np.array([json_dumps(chk)]), **save)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\ndef json_dumps(o):\n    import json\n    return json.dumps(o, ensure_ascii=False)\n\n\n@torch.no_grad()\ndef self_test_score(device):\n    """小さなランダムのモデルで：KV キャッシュで 1 歩ずつ出した ℓ・エントロピーと、読み直しの値が合う。U が −log p₁ と合う。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4, [20, 21, 22, 23] * 4, [30, 31, 32, 33] * 4]\n    L, T = 40, 0.7\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    g = torch.Generator(device=device).manual_seed(0)\n    E, Ent, U, Y = [], [], [], []\n    for t in range(L):\n        ell, _, ent, _ = step_stats(logits, T, valid, 33)\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        nxt = torch.multinomial(torch.softmax(z / T, -1), 1, generator=g)\n        U.append(-torch.log_softmax(z, -1).gather(-1, nxt)[:, 0]); E.append(ell); Ent.append(ent); Y.append(nxt[:, 0])\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda xs: torch.stack(xs, 1).double().cpu().numpy()\n    toks = torch.stack(Y, 1).cpu().numpy()\n    r = score_seqs(model, prompts, toks, np.full(len(prompts), T), valid, 33, device)\n    assert np.abs(r[\'ELL\'] - st(E)).max() < 1e-3 and np.abs(r[\'HT\'] - st(Ent)).max() < 1e-3 and np.abs(r[\'U\'] - st(U)).max() < 1e-3\n    # ずらすと合わない（確かめの仕組みが効く）\n    r2 = score_seqs(model, prompts, np.roll(toks, 1, axis=1), np.full(len(prompts), T), valid, 33, device)\n    assert np.abs(r2[\'U\'] - st(U)).max() > 1e-2\n    return \'OK\'\n\n\n# ---------- 分析（CPU） ----------\n\ndef phases(c, exit_th=A8_EXIT):\n    """写しの印 c（長さ L）→ 歩 16〜L−16 の相（0 ＝ 液体、1 ＝ 結晶）と、その歩の番号。入口 0.5、出口 exit_th の履歴つき。"""\n    L = len(c)\n    r = np.convolve(c.astype(np.float64), np.ones(A8_W), \'valid\') / A8_W   # r[k] は歩 k..k+31 の平均 → 中心の歩 k+16\n    t = np.arange(len(r)) + A8_W // 2\n    s = np.zeros(len(r), np.int8)\n    cur = 1 if r[0] >= A8_ENTER - 1e-12 else 0\n    for i, v in enumerate(r):\n        if cur == 0 and v >= A8_ENTER - 1e-12:\n            cur = 1\n        elif cur == 1 and v < exit_th - 1e-12:\n            cur = 0\n        s[i] = cur\n    return s, t\n\n\ndef doc_counts(C, U, HT, exit_th=A8_EXIT):\n    """C・U・HT：(S, L)（同じ記事・同じ温度のサンプル）→ [N0, n01, N1, n10, ΣU液, n液, ΣU結, n結, ΣHT液, ΣHT結]。"""\n    acc = np.zeros(10)\n    for s in range(C.shape[0]):\n        st, t = phases(C[s], exit_th)\n        a, b = st[:-1], st[1:]\n        acc[0] += (a == 0).sum(); acc[1] += ((a == 0) & (b == 1)).sum()\n        acc[2] += (a == 1).sum(); acc[3] += ((a == 1) & (b == 0)).sum()\n        liq, cry = st == 0, st == 1\n        acc[4] += U[s, t[liq]].sum(); acc[5] += liq.sum(); acc[6] += U[s, t[cry]].sum(); acc[7] += cry.sum()\n        acc[8] += HT[s, t[liq]].sum(); acc[9] += HT[s, t[cry]].sum()\n    return acc\n\n\ndef build_counts(Z, exit_th=A8_EXIT):\n    """Z：読み直しの npz（dict でもよい）→ (17 の温度, nD, 10)。同じ温度は 3 つの材料を足す。"""\n    nD = len(Z[\'use\'])\n    out = np.zeros((len(A8_TU), nD, 10))\n    for key, _, temps in A8_SOURCES:\n        toks, U, HT = Z[f\'{key}_toks\'], Z[f\'{key}_U\'], Z[f\'{key}_HT\']\n        for k, T in enumerate(temps):\n            ku = A8_TU.index(T)\n            for j in range(nD):\n                C = np.stack([copy_marks(toks[k, s, j]) for s in range(toks.shape[1])])\n                out[ku, j] += doc_counts(C, U[k, :, j].astype(np.float64), HT[k, :, j].astype(np.float64), exit_th)\n    return out\n\n\ndef fit_vh(tot, temps_use=None):\n    """tot：(17, 10)（記事を足したもの）→ 使う温度・ln K・ファントホッフの当てはめ・熱量計の ΔH。"""\n    n01, n10, N0, N1 = tot[:, 1], tot[:, 3], tot[:, 0], tot[:, 2]\n    ok = (n01 >= A8_MIN_EV) & (n10 >= A8_MIN_EV) if temps_use is None else np.isin(np.arange(len(A8_TU)), temps_use)\n    idx = np.flatnonzero(ok)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        lnK = np.log(n01 / N0) - np.log(n10 / N1)\n        dHcal = tot[:, 4] / tot[:, 5] - tot[:, 6] / tot[:, 7]\n        dScal = tot[:, 8] / tot[:, 5] - tot[:, 9] / tot[:, 7]\n    res = dict(use_idx=idx.tolist(), lnK=lnK.tolist(), dHcal_T=dHcal.tolist(), dScal_T=dScal.tolist())\n    if len(idx) < A8_MIN_TEMPS or not np.all(np.isfinite(lnK[idx])) or not np.all(np.isfinite(dHcal[idx])):\n        res.update(ok=False)\n        return res\n    x = 1.0 / np.array(A8_TU)[idx]; y = lnK[idx]\n    w = 1.0 / (1.0 / n01[idx] + 1.0 / n10[idx])\n    X = np.stack([np.ones_like(x), x], 1)\n    a, b = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * y))\n    Tm = (-b / a) if (b > 0 and a < 0) else float(\'nan\')\n    dH_cal = float(np.mean(dHcal[idx]))\n    n = b / dH_cal if dH_cal > 0 else float(\'nan\')\n    res.update(ok=True, dH_vH=float(b), dS_vH=float(-a), Tm=float(Tm), dH_cal=dH_cal, n=float(n),\n               dS_cal=float(np.mean(dScal[idx])), Tcal_ratio=float(dH_cal / np.mean(dScal[idx])) if np.mean(dScal[idx]) > 0 else float(\'nan\'))\n    if len(idx) >= 4:                                                      # 曲がり（記録）：2 次の係数\n        X2 = np.stack([np.ones_like(x), x, x * x], 1)\n        res[\'quad\'] = np.linalg.solve(X2.T @ (w[:, None] * X2), X2.T @ (w * y)).tolist()\n    return res\n\n\ndef q1_word(Tm_ci, T0):\n    t0, lo0, hi0 = T0\n    lo, hi = Tm_ci\n    if not (np.isfinite(lo) and np.isfinite(hi)):\n        return \'はっきりしない\'\n    if lo > hi0:\n        return \'ずれる（高い）\'\n    if hi < lo0:\n        return \'ずれる（低い）\'\n    if lo >= t0 / A8_Q1_BAND and hi <= t0 * A8_Q1_BAND:\n        return \'合う\'\n    return \'はっきりしない\'\n\n\ndef q2_word(n):\n    if not np.isfinite(n):\n        return \'はっきりしない\'\n    if n <= 0:\n        return \'向きが逆\'\n    for edge, w in A8_Q2_BINS:\n        if n < edge:\n            return w\n\n\ndef analyze_model(cnt, T0, seed=A8_BOOT_SEED):\n    """cnt：(17, nD, 10) → 点・選び直しの区間・Q1 の言葉。"""\n    nD = cnt.shape[1]\n    pt = fit_vh(cnt.sum(1))\n    row = dict(point=pt, temps=A8_TU)\n    tot = cnt.sum(1)\n    row[\'by_T\'] = [dict(T=T, n_freeze=int(tot[k, 1]), n_melt=int(tot[k, 3]), steps_liq=int(tot[k, 0]), steps_cry=int(tot[k, 2]),\n                        k_f=float(tot[k, 1] / tot[k, 0]) if tot[k, 0] else None, k_m=float(tot[k, 3] / tot[k, 2]) if tot[k, 2] else None)\n                   for k, T in enumerate(A8_TU)]\n    if not pt[\'ok\']:\n        row.update(Q1_word=\'はっきりしない\', ci=None, boot_fail=1.0)\n        return row\n    rng = np.random.default_rng(seed)\n    keep, fail = {k: [] for k in (\'dH_vH\', \'Tm\', \'dH_cal\', \'n\')}, 0\n    for _ in range(A8_BOOT):\n        w = np.bincount(rng.integers(0, nD, nD), minlength=nD).astype(float)\n        r = fit_vh(np.tensordot(cnt, w, axes=([1], [0])), temps_use=pt[\'use_idx\'])\n        if not r[\'ok\'] or not np.isfinite(r[\'Tm\']):\n            fail += 1\n            continue\n        for k in keep:\n            keep[k].append(r[k])\n    ff = fail / A8_BOOT\n    ci = {k: ([float(np.percentile(v, 2.5, method=\'lower\')), float(np.percentile(v, 97.5, method=\'higher\'))] if len(v) >= 20 else [float(\'nan\')] * 2)\n          for k, v in keep.items()}\n    row.update(ci=ci, boot_fail=ff)\n    row[\'Q1_word\'] = q1_word(ci[\'Tm\'], T0) if ff <= 0.1 and np.isfinite(pt[\'Tm\']) else \'はっきりしない\'\n    return row\n\n\ndef analyze8(Zs):\n    """Zs：{モデル: 読み直しの npz}。→ まとめ。"""\n    res = {\'models\': {}, \'temps\': A8_TU}\n    for name, _ in A8_MODELS:\n        Z = Zs[name]\n        cnt = build_counts(Z)\n        row = analyze_model(cnt, A8_T0[name])\n        row[\'T0_ref\'] = A8_T0[name]\n        row[\'sens_exit\'] = {str(e): fit_vh(build_counts(Z, e).sum(1)) for e in A8_SENS_EXIT}\n        # 確かめ（記録）：温度 1 の生成では、平均の U と平均の H1 は期待値で等しい\n        k1 = A8_T22.index(1.0)\n        row[\'check_T1_U_vs_H1\'] = [float(Z[\'a22_U\'][k1].mean()), float(Z[\'a22_H1\'][k1].mean())]\n        row[\'score_check\'] = str(Z[\'check\'][0]) if \'check\' in Z else None\n        res[\'models\'][name] = row\n    res[\'Q1\'] = sum(res[\'models\'][m][\'Q1_word\'] == \'合う\' for m, _ in A8_MODELS)\n    ns = [res[\'models\'][m][\'point\'].get(\'n\', float(\'nan\')) for m, _ in A8_MODELS]\n    ns = [v for v in ns if np.isfinite(v)]\n    res[\'Q2_n_each\'] = ns\n    res[\'Q2\'] = q2_word(float(np.median(ns))) if len(ns) >= 2 else \'はっきりしない\'\n    return res\n\n\ndef self_test_a8():\n    """作り物：2 つの状態を行き来する鎖を、凍る速さ k_f ＝ exp(a_f − E_f/T)・溶ける速さ k_m で作る。K ＝ k_f/k_m の\n    ΔH ＝ E_m − E_f を当てはめで取り戻せるか、T_m が合うか、U の差（熱量計）から n が出るか。"""\n    rng = np.random.default_rng(0)\n    L, nD, S = 512, 40, 6\n    dH_true, Tm_true = 6.0, 0.8                    # ln K ＝ dH/T − dH/Tm\n    Uc, Ul = 0.2, 3.2                              # 熱量計の ΔH ＝ 3.0 → n ＝ 2\n    Z = {\'use\': np.arange(nD), \'check\': np.array([\'{}\'])}\n    for key, _, temps in A8_SOURCES:\n        nT = len(temps)\n        toks = np.zeros((nT, 2, nD, L), np.int64); U = np.zeros((nT, 2, nD, L), np.float32)\n        for k, T in enumerate(temps):\n            lnK = dH_true / T - dH_true / Tm_true\n            km = 0.01 * math.exp(-lnK / 2); kf = 0.01 * math.exp(lnK / 2)\n            for s in range(2):\n                for j in range(nD):\n                    st, seq, u = 0, [], []\n                    period = list(rng.integers(100, 30000, 7))\n                    for t in range(L):\n                        if t > 0:\n                            st = (1 if rng.random() < kf else 0) if st == 0 else (0 if rng.random() < km else 1)\n                        if st == 1:\n                            seq.append(period[t % 7]); u.append(Uc)\n                        else:\n                            seq.append(int(rng.integers(100, 30000))); u.append(Ul)\n                    toks[k, s, j] = seq; U[k, s, j] = u\n        Z[f\'{key}_toks\'] = toks; Z[f\'{key}_U\'] = U; Z[f\'{key}_HT\'] = U.copy(); Z[f\'{key}_H1\'] = U.copy()\n    row = analyze_model(build_counts(Z), (0.8, 0.78, 0.82))\n    p = row[\'point\']\n    assert p[\'ok\'] and abs(p[\'dH_vH\'] - dH_true) / dH_true < 0.35, p\n    assert abs(p[\'Tm\'] - Tm_true) < 0.06, p\n    assert 2.0 < p[\'dH_cal\'] < 3.1 and 1.2 < p[\'n\'] < 3.2, p\n    # 言葉\n    assert q1_word([0.79, 0.83], (0.8, 0.78, 0.82)) == \'合う\' and q1_word([0.9, 0.95], (0.8, 0.78, 0.82)) == \'ずれる（高い）\'\n    assert q1_word([0.6, 0.7], (0.8, 0.78, 0.82)) == \'ずれる（低い）\' and q1_word([0.5, 1.1], (0.8, 0.78, 0.82)) == \'はっきりしない\'\n    assert q2_word(1.1) == \'1 くらい（ばらばらに溶ける）\' and q2_word(35) == \'数十トークン（10〜100、結晶核くらい）\' and q2_word(-2) == \'向きが逆\'\n    # 相：はっきりした結晶の区間を拾う\n    c = np.zeros(300, np.int8); c[100:200] = 1\n    s, t = phases(c)\n    assert s[t == 90].item() == 0 and s[t == 150].item() == 1 and s[t == 250].item() == 0\n    return \'OK\'\n'
assert hashlib.sha256(SRC8.encode()).hexdigest() == '06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd'
exec(SRC22)
exec(SRCON)
exec(SRC8)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 読み直し（モデルごと）→ 分析 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_onset(), self_test_a8(), self_test_score('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_8', exist_ok=True)
    for name, hf_id in A8_MODELS:
        run_score(name, hf_id, '/content/drive/MyDrive/logit-thermo', '/content/drive/MyDrive/logit-thermo/A2_8', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    Zs = {name: dict(np.load('/content/drive/MyDrive/logit-thermo/A2_8' + f'/{name}.npz')) for name, _ in A8_MODELS}
    r = analyze8(Zs)
    r.update(gate='docs/A2_8_GATE.md', a2_8_sha256='06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd', a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', onset_sha256='0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9',
             torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
             gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision())
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_8' + '/a2_8_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-8（ファントホッフと熱量計）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-8 の判定を A2_8/a2_8_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
